In [ ]:
import torch

torch.manual_seed(43)


# What do we want to estimate/predict?

$y$ is our target. We want our model to estimate $y$ given $x$.

For each example in our dataset, we have an input and a training target. 

E.g., our dataset of size 2, has two pairs of inputs and targets.

In [ ]:
batch_size = 2
x = torch.randn(batch_size, 5)


In [ ]:
x

In [ ]:

x.shape


In [ ]:
y = torch.rand(batch_size, 3)


In [ ]:
y

In [ ]:
y.shape

# Our model

$$
\hat{y} = xW + b
$$

A multivariate, multi-output linear regression model.

W and b will be our learnable parameters, which we can represent as:

$$
\theta = \{W, b\}
$$

# requires_grad=True

This records the mathematical operations during forward so that gradients can be computed during backward propagation.

In [ ]:
from torch import nn

w = nn.Parameter(torch.randn(5, 3))
b = nn.Parameter(torch.randn(3))
z = torch.matmul(x, w) + b


# What does this neural network look like?

![Diagram of the 5-input, 3-output fully connected network](network-architecture.png)

In [ ]:
w

In [ ]:
w.shape

In [ ]:
w.grad

In [ ]:
w.grad == None

In [ ]:
b

In [ ]:
b.shape

In [ ]:
b.grad

In [ ]:
z

In [ ]:
z.grad

In [ ]:
z.shape

# What is the error/difference/loss/cost between our output and target?

This tells us how "far away" our model is from correctly estimating the target for the given input.

E.g., we could use "mean squared error" between the output and the target as the loss.

$$
  \mathcal{L}
  = \frac{1}{N}\sum_{i=1}^{N}
  \left(
  \frac{1}{D}\sum_{j=1}^{D}(z_{ij}-y_{ij})^2
  \right)
  $$



In [ ]:
loss = ((z - y)**2).mean(dim=1).mean(dim=0)
loss

# Time to learn: Lets update our parameters to minimise the loss


The goal is to find parameters $\theta$ that minimize a loss function:

$$
\theta^* = \arg\min_{\theta} \mathcal{L}(\theta)
$$

  First, calculate the gradient of the loss with respect to the parameters:

  $$
  \nabla_{\theta}\mathcal{L}(\theta)
  $$

  
Later, we will use this to update our parameters.

In [ ]:
w.grad

In [ ]:
loss.backward()

In [ ]:
w.grad

# Optimizer

Then update the parameters in the direction opposite to the gradient:

  $$
  \theta_{t+1}
  =
  \theta_t
  -
  \eta \nabla_{\theta}\mathcal{L}(\theta_t)
  $$

  where:

  - $\theta_t$ represents the current model parameters, such as weights and
  biases
  - $\eta$ is the learning rate
  - $\nabla_{\theta}\mathcal{L}(\theta_t)$ is the gradient of the loss

  For a weight $w$ and bias $b$, the updates are:

  $$
  w_{t+1}
  =
  w_t
  -
  \eta \frac{\partial \mathcal{L}}{\partial w}
  $$

  $$
  b_{t+1}
  =
  b_t
  -
  \eta \frac{\partial \mathcal{L}}{\partial b}
  $$


![](./gradient_descent-2.png)

In [ ]:
w

In [ ]:
learning_rate = 1e-2

with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

In [ ]:
w

# Zero your gradients!

Now, we just updated our parameters based on the gradients. 

We have to reset our gradients for the next update.

In [ ]:
w.grad

In [ ]:
w.grad.zero_()

# Training over a large dataset

Before our training data had two examples. 

Instead, lets use a large dataset, and iterate over it in mini-batches.

We need the mini-batches as the dataset is too large to train the model all in one go.

So, as opposed to gradient descent, where every epoch (which means training over your entire dataset) in one go, we will do stochastic gradient descent.

#### Stochastic gradient descent: performing gradient descent on mini-batches of your dataset, rather than your whole dataset.

Therefore, one epoch will consist of multiple mini-batches; your model will be updated mulitple times per epoch.

In [ ]:
from tqdm import tqdm

# Define our hyperparameters:
training_size = 1000
number_of_epochs = 10
learning_rate = 0.001
mini_batch_size = 10

inputs = torch.randn(training_size, 5)
targets = torch.tensor([0.1, 10.0, 1000.0]).repeat(training_size, 1)

for epoch in range(number_of_epochs):

    progress_bar = tqdm(range(training_size), desc=f"Epoch {epoch + 1}")

    for i in progress_bar:
        x = inputs[i:i+mini_batch_size]
        y = targets[i:i+mini_batch_size]
        
        z = torch.matmul(x, w) + b
        loss = ((z - y)**2).mean(dim=1).mean(dim=0)
        
        loss.backward()
        
        with torch.no_grad():
            w -= learning_rate * w.grad
            b -= learning_rate * b.grad
        
        w.grad.zero_()
        b.grad.zero_()

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

In [ ]:
torch.matmul(x, w) + b

In [ ]:
torch.set_printoptions(sci_mode=False, precision=4)
torch.matmul(x, w) + b

# A feedforward neural network layer

$\hat{y} = xW + b$

In [ ]:
# Before
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w) + b

# After
layer = torch.nn.Linear(5, 3)

In [ ]:
layer

In [ ]:
layer(x)  # Equivalent to z = torch.matmul(x, w) + b

# torch.nn.Module: How we actually construct models

This allows us to:
- Define the learnable parameters in __init__
- Have a callable forward propagation function/definition.

In [ ]:
class AffineTransform(nn.Module):
    def __init__(self, in_features, out_features, bias=True):

        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.bias = bias

        self.w = nn.Parameter(torch.randn(self.in_features, self.out_features, requires_grad=True))
        if bias:
            self.b = nn.Parameter(torch.randn(self.out_features, requires_grad=True))

    def forward(self, x):
        if self.bias:
            return torch.matmul(x, self.w) + self.b
        else:
            return torch.matmul(x, self.w)

    def extra_repr(self):
        return (
            f"in_features={self.in_features}, "
            f"out_features={self.out_features}, "
            f"bias={self.bias}"
        )

In [ ]:
same_layer = AffineTransform(5, 3)


In [ ]:
layer

In [ ]:
same_layer

# Finally... something deep

In [ ]:
class FeedforwardNeuralNetwork(nn.Module):
    def __init__(self, in_features, hidden_features, out_features, num_hidden_layers):
        super().__init__()

        layers = []

        for layer_index in range(num_hidden_layers):
            input_size = in_features if layer_index == 0 else hidden_features

            layers.extend([
                nn.Linear(input_size, hidden_features),
                nn.ReLU(),  # This is the activation function applied after each hidden layer
            ])

        layers.append(nn.Linear(hidden_features, out_features))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
model = FeedforwardNeuralNetwork(5, 10, 3, 2)
model

![](./featured.jpg)

# What is an activation function? 

We add an activation function to our layers to make them non-linear; this allows us to learn more complex functions.

$\hat{y} = \sigma(xW + b)$

![](./activations.png)

# Torch has classes for the loss and the optimiser as well

In [ ]:
loss = torch.nn.functional.mse_loss(z, y) 

optimiser = torch.optim.SGD(model.parameters(), lr=0.001)

In [ ]:
params = list(model.parameters())
print(params)

In [ ]:
for name, param in model.named_parameters():
    print(name, param.shape)

In [ ]:
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(total)

In [ ]:
from tqdm import tqdm

# Define our hyperparameters:
training_size = 1000
number_of_epochs = 10
learning_rate = 0.001
mini_batch_size = 10

model = FeedforwardNeuralNetwork(5, 10, 3, 2)
optimiser = torch.optim.SGD(model.parameters(), lr=learning_rate)

inputs = torch.randn(training_size, 5)
targets = torch.tensor([0.1, 10.0, 1000.0]).repeat(training_size, 1)

for epoch in range(number_of_epochs):

    progress_bar = tqdm(range(training_size), desc=f"Epoch {epoch + 1}")

    for i in progress_bar:
        x = inputs[i:i+mini_batch_size]
        y = targets[i:i+mini_batch_size]
        
        z = model(x)
        loss = torch.nn.functional.mse_loss(z, y)
        
        loss.backward()
        optimiser.step()
        optimiser.zero_grad()

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

In [ ]:
model(x)

# Lets use real data: Grayscale images of handwritten digits (0 through 9)

In [ ]:
from torchvision.datasets import MNIST
from torch.utils.data import Dataset


class MyMNISTDataset(Dataset):
    def __init__(self, root="./data", train=True):
        mnist = MNIST(
            root=root,
            train=train,
            download=True,
        )

        self.images = mnist.data
        self.labels = mnist.targets

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = self.images[index]
        label = self.labels[index]

        # Convert uint8 pixels to float32 values in [0, 1]
        image = image.float() / 255.0

        # Flatten [28, 28] into [784]
        image = image.flatten()

        return image, label

In [ ]:
train_dataset = MyMNISTDataset('./data', train=True)

# What does the data look like?

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for index, ax in enumerate(axes.flat):
    image, label = train_dataset[index]

    ax.imshow(image.reshape(28, 28), cmap="gray")
    ax.set_title(f"Label: {label.item()}")
    ax.axis("off")

plt.tight_layout()
plt.show()

# What does the model see?

In [ ]:
image.shape

In [ ]:
image

# What is it estimating?

In [ ]:
label

# How many training examples are there?

In [ ]:
len(train_dataset)

In [ ]:
from torch.utils.data import DataLoader, random_split

# Define our hyperparameters:
number_of_epochs = 3
learning_rate = 0.001
mini_batch_size = 32

# Get our torch datasets for training and testing:
full_train_dataset = MyMNISTDataset('./data', train=True)
test_dataset = MyMNISTDataset('./data', train=False)

# Split the training dataset into a training and validation dataset:
split_generator = torch.Generator().manual_seed(42)
train_dataset, validation_dataset = random_split(
    full_train_dataset,
    [59000, 1000],
    generator=split_generator,
)

# Define our dataloaders for training, validation, and testing:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=mini_batch_size,
    shuffle=True,
)
validation_dataloader = DataLoader(
    validation_dataset,
    batch_size=mini_batch_size,
    shuffle=False,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=mini_batch_size,
    shuffle=False,
)

# Define our model:
model = FeedforwardNeuralNetwork(784, 16, 10, 2)

# Define our optimiser:
optimiser = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Train the model:
for epoch in range(number_of_epochs):

    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}")

    model.train()  # Set the model to training mode (enables training features)
    for batch in progress_bar:

        x, y = batch
        
        z = model(x)
        loss = torch.nn.functional.cross_entropy(z, y)
        
        loss.backward()
        optimiser.step()
        optimiser.zero_grad()

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    # Evaluate the model on the validation dataset:
    model.eval()  # Set the model to evaluation mode (disables training features)

    val_accuracy = 0
    val_loss = 0
    for batch in validation_dataloader:
        x, y = batch
        
        with torch.no_grad():  # Disable gradient computation for evaluation
            z = model(x)
            predictions = torch.argmax(z, dim=1)
            val_accuracy += (predictions == y).sum().item()
            val_loss += torch.nn.functional.cross_entropy(z, y).item()
    val_accuracy /= len(validation_dataset)
    val_loss /= len(validation_dataloader)
    print(f"Validation accuracy for epoch {epoch + 1}: {val_accuracy:.4f}")
    print(f"Validation loss for epoch {epoch + 1}: {val_loss:.4f}")    

In [ ]:
# Testing:
model.eval()  # Set the model to evaluation mode (disables training features)

test_accuracy = 0
test_loss = 0
for batch in test_dataloader:
    x, y = batch
    
    with torch.no_grad():  # Disable gradient computation for evaluation
        z = model(x)
        predictions = torch.argmax(z, dim=1)
        test_accuracy += (predictions == y).sum().item()
        test_loss += torch.nn.functional.cross_entropy(z, y).item()
test_accuracy /= len(test_dataset)
test_loss /= len(test_dataloader)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test loss: {test_loss:.4f}")

# Lets see it in action



In [ ]:
model.eval()

x, y = next(iter(test_dataloader))

with torch.no_grad():
    logits = model(x)
    predictions = logits.argmax(dim=1)

number_to_show = 10

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for index, axis in enumerate(axes.flat):
    image = x[index].reshape(28, 28)  # Images were flattened from [28, 28] to [784].


    axis.imshow(image, cmap="gray")
    axis.set_title(
        f"Predicted: {predictions[index].item()}\n"
        f"Actual: {y[index].item()}\n",
        color="green" if predictions[index] == y[index] else "red",
    )
    axis.axis("off")

plt.tight_layout()
plt.show()

# Challenge: who can get the highest test set accuracy?

Hints: look at the hyperparameters, the optimiser, the model.

~90% accuracy: change a few numbers.

~95% accuracy: change a few more numbers.

~98% accuracy: this is very attainable, but it will require investigating Google (hint, Adam).

~99% accuracy: can you possibly implement a convolutional neural network in this amount of time? (Hint, you don't need to implement it yourself.)

Beyond: Did you get help from Claude/Codex?


